In [29]:
import pandas as pd
import altair as alt
from vega_datasets import data
import numpy as np

df = pd.read_csv(r"../assets/data/licenses_fall2022.csv")
df = df.dropna(subset=["Discipline Reason"])

search_terms = {
    "Failed to File and/or Pay": ["file and/or pay"],
    "Failed to Disclose": ["disclose"],
    "Failed to Report": ["report"],
    "Criminal Conviction": ["criminal", "conviction"],
    "Unlicensed": ["unlicensed", "without a license", "prior to licensure", "without a valid"],
    "Fraud": ["falsif", "misstatement", "fraudulent", "inaccurate records", "false statement"],
    "Debt/Adminstrative": ["child support", "delinquent", "loan", "continuing education", "proof of hours", "request for information", "failing to appear"]
}

reasons = df["Discipline Reason"].unique()
categorized_reasons = {}
all_matched_reasons = set()

for category, keywords in search_terms.items():
    matches = [r for r in reasons if any(k in str(r).lower() for k in keywords)]
    
    if category == "Criminal Conviction":
        matches = [m for m in matches if "report" not in str(m).lower()]
    
    categorized_reasons[category] = matches
    all_matched_reasons.update(matches)

categorized_reasons["Other"] = [r for r in reasons if r not in all_matched_reasons]

for cat, matches in categorized_reasons.items():
    print(f"{cat:12} {len(matches)}")

df.head()

Failed to File and/or Pay 32
Failed to Disclose 9
Failed to Report 16
Criminal Conviction 26
Unlicensed   29
Fraud        8
Debt/Adminstrative 16
Other        45


,_id,License Type,Description,License Number,License Status,Business,Title,First Name,Middle,Last Name,...,Specialty/Qualifier,Controlled Substance Schedule,Delegated Controlled Substance Schedule,Ever Disciplined,LastModifiedDate,Case Number,Action,Discipline Start Date,Discipline End Date,Discipline Reason
43,776333,DETECTIVE BOARD,FIREARM CONTROL CARD,229047854.0,TERMINATED CARD RETURNED,N,NaN,SHIRLEY LYN,NaN,KARNATZ,...,NaN,NaN,NaN,Y,08/07/2006,2.001007e+09,Suspension,06/10/2003,NaN,Failed to file and/or pay Illinois income taxes.
48,1035198,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129261059,NOT RENEWED,N,NaN,DOROTHY,C,CRIBBS,...,NaN,NaN,NaN,Y,07/19/2018,2.017010e+09,Suspension,01/05/2018,02/02/2022,Failure to file and/or pay Illinois state inco...
111,1115012,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129353591,NOT RENEWED,N,NaN,NaN,F,NaN,...,NaN,NaN,NaN,Y,07/19/2018,2.012007e+09,Revocation,07/25/2012,11/18/2015,for being more than 30 days delinquent in the ...
138,663182,COSMO,LICENSED NAIL TECHNICIAN,169009359,NOT RENEWED,N,NaN,MAGDALENO,NaN,BARRAGAN,...,NaN,NaN,NaN,N,07/28/2014,1.996001e+09,Suspension,09/03/1996,NaN,Obtained license fraudulently by falsifying in...
165,746349,DENTAL,REGISTERED DENTAL HYGIENIST,020007262,NOT RENEWED,N,NaN,PRISCILLA,NaN,FARINA,...,NaN,NaN,NaN,Y,05/09/2022,2.018007e+09,Suspension,10/06/2018,02/13/2019,Failure to file and/or pay Illinois state inco...


In [30]:
# map categorized reasons
def map_reason(reason):
    for cat, keywords in categorized_reasons.items():
        if reason in keywords: return cat
    return "other"
df['Category'] = df['Discipline Reason'].apply(map_reason)

# calculate discipline length
df['Discipline Start Date'] = pd.to_datetime(df['Discipline Start Date'], errors='coerce')
df['Discipline End Date'] = pd.to_datetime(df['Discipline End Date'], errors='coerce')
df['Discipline Length'] = (df['Discipline End Date'] - df['Discipline Start Date']).dt.days

# clean zip codes and drop floats
df['Zip'] = df['Zip'].astype(str).str.extract(r'^(\d{5})')[0]

# valid state mapping and fips codes
state_fips = {
    'AL': 1, 'AK': 2, 'AZ': 4, 'AR': 5, 'CA': 6, 'CO': 8, 'CT': 9, 'DE': 10,
    'DC': 11, 'FL': 12, 'GA': 13, 'HI': 15, 'ID': 16, 'IL': 17, 'IN': 18,
    'IA': 19, 'KS': 20, 'KY': 21, 'LA': 22, 'ME': 23, 'MD': 24, 'MA': 25,
    'MI': 26, 'MN': 27, 'MS': 28, 'MO': 29, 'MT': 30, 'NE': 31, 'NV': 32,
    'NH': 33, 'NJ': 34, 'NM': 35, 'NY': 36, 'NC': 37, 'ND': 38, 'OH': 39,
    'OK': 40, 'OR': 41, 'PA': 42, 'RI': 44, 'SC': 45, 'SD': 46, 'TN': 47,
    'TX': 48, 'UT': 49, 'VT': 50, 'VA': 51, 'WA': 53, 'WV': 54, 'WI': 55, 'WY': 56
}

# geographic centers for states 
state_centers = {
    'AL': [32.7794, -86.8287], 'AK': [64.0685, -152.2782], 'AZ': [34.2744, -111.6602],
    'AR': [34.8938, -92.4426], 'CA': [37.1841, -119.4696], 'CO': [38.9972, -105.5478],
    'CT': [41.6219, -72.7273], 'DE': [38.9896, -75.5050], 'DC': [38.9101, -77.0147],
    'FL': [28.6305, -82.4497], 'GA': [32.6415, -83.4426], 'HI': [20.2927, -156.3737],
    'ID': [44.3509, -114.6130], 'IL': [40.0417, -89.1965], 'IN': [39.8942, -86.2816],
    'IA': [42.0751, -93.4960], 'KS': [38.4985, -98.3200], 'KY': [37.5347, -85.3021],
    'LA': [31.0689, -91.9968], 'ME': [45.3695, -69.2428], 'MD': [39.0457, -76.7909],
    'MA': [42.2596, -71.8083], 'MI': [44.3467, -85.4102], 'MN': [46.2807, -94.3053],
    'MS': [32.7364, -89.6678], 'MO': [38.3566, -92.4580], 'MT': [47.0527, -109.6333],
    'NE': [41.5378, -99.7951], 'NV': [39.3289, -116.6312], 'NH': [43.6805, -71.5811],
    'NJ': [40.1907, -74.6728], 'NM': [34.4071, -106.1126], 'NY': [42.9538, -75.5268],
    'NC': [35.5586, -79.3877], 'ND': [47.4501, -100.4659], 'OH': [40.2862, -82.7937],
    'OK': [35.5889, -97.4943], 'OR': [43.9336, -120.5583], 'PA': [40.8781, -77.7996],
    'RI': [41.6762, -71.5562], 'SC': [33.9169, -80.8964], 'SD': [44.4443, -100.2263],
    'TN': [35.8580, -86.3505], 'TX': [31.4757, -99.3312], 'UT': [39.3055, -111.6703],
    'VT': [44.0687, -72.6658], 'VA': [37.5215, -78.8537], 'WA': [47.3826, -120.4472],
    'WV': [38.6409, -80.6227], 'WI': [44.6243, -89.9941], 'WY': [42.9957, -107.5512]
}

# clean invalid states
df = df[df['State'].isin(state_fips.keys())].copy()

# convert centers dictionary to dataframe
df_centers = pd.DataFrame.from_dict(state_centers, orient='index', columns=['lat_center', 'lon_center']).reset_index()
df_centers = df_centers.rename(columns={'index': 'State'})

# prepare geo data
zipcodes = data.zipcodes()
df_geo = pd.merge(df, zipcodes[['zip_code', 'latitude', 'longitude']], left_on='Zip', right_on='zip_code', how='inner')

# strictly merge the precise geographic centers
df_geo = pd.merge(df_geo, df_centers, on='State', how='left')

# add fips IDs for background mapping
df_geo['id'] = df_geo['State'].map(state_fips)
active_states = df_geo[['id']].dropna().drop_duplicates()
active_states['active'] = 1

In [ ]:
# custom color palette
custom_colors = ["#48bf8e", "#f53176", "#27d53d", "#e456d8", "#2a6b2a", "#8d1993", "#95b833", "#1f1b5f", "#e9ad6f", "#0b29d0", "#96b299", "#3f0116", "#5ac4f8", "#863c2c"]
"""
@article{gramazio-2017-ccd,
  author={Gramazio, Connor C. and Laidlaw, David H. and Schloss, Karen B.},
  journal={IEEE Transactions on Visualization and Computer Graphics},
  title={Colorgorical: creating discriminable and preferable color palettes for information visualization},
  year={2017}
}
"""

# extract license types and match them to custom colors
active_license_types = list(df_geo['License Type'].dropna().unique())
color_range = (custom_colors * (len(active_license_types) // len(custom_colors) + 1))[:len(active_license_types)]

# parameters
license_dropdown = alt.binding_select(options=[None] + active_license_types, labels=['All'] + active_license_types, name='License Type: ')
license_selection = alt.selection_point(fields=['License Type'], bind=license_dropdown)

disciplined_dropdown = alt.binding_select(options=[None, 'Y', 'N'], labels=['All', 'Y', 'N'], name='Ever Disciplined: ')
disciplined_selection = alt.selection_point(fields=['Ever Disciplined'], bind=disciplined_dropdown)

group_dropdown = alt.binding_select(options=['Zipcode', 'State'], name='Group By: ')
group_param = alt.param(name='GroupView', bind=group_dropdown, value='Zipcode')

size_slider = alt.binding_range(min=1, max=100, step=1, name='Dot Size Scale: ')
size_param = alt.param(name='DotScale', bind=size_slider, value=20)

# map layers
cols_1 = ['City', 'State', 'Zip', 'License Type', 'Ever Disciplined', 'latitude', 'longitude', 'lat_center', 'lon_center', 'id']
df_chart1 = df_geo[cols_1].dropna(subset=['latitude', 'longitude', 'lat_center', 'lon_center']).replace({np.nan: None})

states = alt.topo_feature(data.us_10m.url, 'states')
background = alt.Chart(states).mark_geoshape(fill='#eeeeee', stroke='white')
highlight = alt.Chart(states).mark_geoshape(fill='#ccebc5', stroke='white').transform_lookup(
    lookup='id', from_=alt.LookupData(active_states, 'id', ['active'])
).transform_filter(alt.datum.active == 1)

# zipcode layer
zip_layer = alt.Chart(df_chart1).mark_circle(opacity=0.8).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    size=alt.Size('scaled_count:Q', scale=None, legend=None), 
    
    color=alt.Color('License Type:N', 
                    scale=alt.Scale(domain=active_license_types, range=color_range), 
                    legend=alt.Legend(title="License Type")),
                    
    tooltip=['City:N', 'State:N', 'Zip:N', 'License Type:N', 'count:Q']
).transform_filter(
    "GroupView == 'Zipcode'"
).transform_filter(
    license_selection
).transform_filter(
    disciplined_selection
).transform_aggregate(
    count='count()',
    groupby=['City', 'State', 'Zip', 'License Type', 'latitude', 'longitude']
).transform_calculate(
    scaled_count="clamp(datum.count * DotScale, 10, 1500)"
)

# state layer
state_layer = alt.Chart(df_chart1).mark_circle(opacity=0.8, color='#2c7bb6').encode(
    longitude='lon_center:Q',
    latitude='lat_center:Q',
    size=alt.Size('scaled_count:Q', scale=None, legend=None),
    tooltip=['State:N', 'count:Q']
).transform_filter(
    "GroupView == 'State'"
).transform_filter(
    license_selection 
).transform_filter(
    disciplined_selection
).transform_aggregate(
    count='count()',
    groupby=['State', 'lat_center', 'lon_center']
).transform_calculate(
    scaled_count="clamp(datum.count * DotScale, 50, 4000)"
)

# combine and project
chart1 = alt.layer(
    background, highlight, zip_layer, state_layer
).project(
    'albersUsa'
).add_params(
    license_selection, disciplined_selection, group_param, size_param
).properties(
    width=800, height=500, title="License Distributions"
)

chart1.save('../assets/json/hw5_chart1.json')
chart1.save('../assets/pngs/hw5_chart1.png')
chart1

alt.LayerChart(...)

In [32]:
# strictly filter data first to fix empty dropdown options and remove negative lengths
df_p2 = df.dropna(subset=['Discipline Length', 'Category']).copy()
df_p2 = df_p2[df_p2['Discipline Length'] >= 0]

# parameters based on cleaned data
cols_2 = ['Category', 'Discipline Reason', 'Discipline Length', 'License Status', 'License Type', 'Business Name']
df_chart2 = df_p2[cols_2].dropna(subset=['Discipline Length']).replace({np.nan: None})

status_types = [None] + list(df_chart2['License Status'].unique())
status_dropdown = alt.binding_select(options=status_types, labels=['All'] + status_types[1:], name='License Status: ')
status_selection = alt.selection_point(fields=['License Status'], bind=status_dropdown)

chart_dropdown = alt.binding_select(options=['Scatter', 'Boxplot'], name='Chart Type: ')
chart_param = alt.param(name='ChartType', bind=chart_dropdown, value='Scatter')

# base layer
base = alt.Chart(df_chart2).encode(
    x=alt.X('Category:N', title='Discipline Reason'),
    y=alt.Y('Discipline Length:Q', title='Discipline Length (Days)'),
    color=alt.Color('Category:N', legend=None)
).add_params(
    status_selection, chart_param
).transform_filter(
    status_selection 
)

# scatter layer
scatter = base.mark_circle(size=60).encode(
    tooltip=['Business Name', 'License Type', 'License Status', 'Discipline Reason', 'Discipline Length']
).transform_filter("ChartType == 'Scatter'")

# boxplot layer
boxplot = base.mark_boxplot(extent='min-max').transform_filter("ChartType == 'Boxplot'")

# combine
chart2 = (scatter + boxplot).properties(
    width=700, 
    height=400,
    title="Discipline Length by Category"
).interactive()

chart2.save('../assets/json/hw5_chart2.json')
chart2.save('../assets/pngs/hw5_chart2.png')
chart2

alt.LayerChart(...)